# Epigenetic marks: ImmuVis trained on Jackson with H3K27me3 and pHH3 (training kernel)

**Question:** can a generative image model infer Jackson's epigenetic marks from the channels both cohorts share?
- **H3K27me3** (`histone_h3_trimethylate`, La139): the repressive chromatin mark.
- **pHH3** (`histone_h3_phospho`, Eu153): histone H3 phosphorylation, a mitosis mark.

**What this kernel does:** trains the published ImmuVis code from scratch on the Jackson train patients, as in
`immuvis-jackson-train`, with two more marker tokens. Scoring is in `epi-immuvis-infer` and `epi-relations`.
- **Channels (16 tokens):** H3, DNA1, DNA2, the 11 shared markers, H3K27me3, pHH3.
- **Same as `immuvis-jackson-train`:** ImmuVis-Conv architecture; arcsinh(x/5) → Butterworth low-pass (0.2) → clip 2.3 → /2.3;
  published masking and loss (channel-subset sampling ≥ 75%, up to 50% channels masked, 60% spatial masking with 8-px patches,
  β-NLL with β = 0.5); AdamW 5e-4 → 1e-5 cosine with 10% warm-up; fp16; 180-min budget; cuDNN off;
  the same Jackson train / val patients (replicate 0 of `train-test-grid`).
- **Checkpoint selection (the only change):** mean pixel r on Jackson val crops of H3K27me3 and pHH3, each predicted from the
  14 shared channels only (the test-time task, since Danenberg has neither mark). The leave-one-marker-out r of the 11 shared
  markers is logged as a check against `immuvis-jackson-train` (0.646).

In [ ]:
import os, glob, json, time, sys, subprocess, warnings, gc, signal
import numpy as np, pandas as pd
from joblib import Parallel, delayed
warnings.filterwarnings("ignore")

IN = os.environ.get("IN_ROOT", "/kaggle/input")
OUT = os.environ.get("OUT_DIR", "/kaggle/working")
SEED = 20261013          # seed of immuvis-jackson-train: identical Jackson split and training seed
MIN_CELLS = 200
COF = 5.0
NUC = ["histone_h3", "dna1", "dna2"]
J_CHAN = ['histone_h3', 'histone_h3_trimethylate', 'cytokeratin_5', 'fibronectin', 'cytokeratin_19', 'cytokeratin_8_18',
          'twist', 'cd68', 'keratin_14_krt14', 'sma', 'vimentin', 'c_myc', 'c_erb_b_2_her2', 'cd3', 'histone_h3_phospho',
          'slug', 'estrogen_receptor_alpha', 'progesterone_receptor_a_b', 'p53', 'cd44', 'cd45', 'gata3', 'cd20',
          'carbonic_anhydrase_ix', 'e_cadherin_p_cadherin', 'ki_67', 'egfr', 's6', 'v_wf', 'm_tor', 'cytokeratin_7',
          'pan_cytokeratin', 'cleaved_parp', 'dna1', 'dna2']
MARKERS = ['c_erb_b_2_her2', 'cd20', 'cd3', 'cd45', 'cd68', 'cytokeratin_5', 'cytokeratin_8_18',
           'estrogen_receptor_alpha', 'ki_67', 'pan_cytokeratin', 'sma']      # sorted, as task_names in immuvis-jackson-train
EPI = ["histone_h3_trimethylate", "histone_h3_phospho"]
CH = NUC + MARKERS + EPI                 # 16 tokens; the first 14 are the shared channels
assert MARKERS == sorted(MARKERS) and all(c in J_CHAN for c in CH)
T0 = time.time()

def tick(msg):
    print(f"[{(time.time() - T0) / 60:6.1f} min] {msg}", flush=True)

def g(pattern):
    return sorted(glob.glob(f"{IN}/**/{pattern}", recursive=True))

## Jackson split (replicate 0 of `train-test-grid`, as in `immuvis-jackson-train`)

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "imagecodecs==2024.9.22"], check=True)
import tifffile
import pyarrow.parquet as pq

cpath = [p for p in g("cells.parquet") if "metabric_id" not in pq.read_schema(p).names]
assert len(cpath) == 1, cpath
cells = pd.read_parquet(cpath[0], columns=["image_name", "object_number", "patient_id", "is_tumour"])
cells = cells[cells.is_tumour == 1]
n_per = cells.groupby("image_name").size()
cells = cells[cells.image_name.isin(n_per.index[n_per >= MIN_CELLS])]
cells = cells.sort_values(["image_name", "object_number"]).reset_index(drop=True)
PAT = cells.patient_id.astype(str).values
jp, jcode = np.unique(PAT, return_inverse=True)
_u = np.random.default_rng(SEED + 11 + 0).random(len(jp))
J_SPLIT = np.where(_u < 0.7, "train", np.where(_u < 0.8, "val", "test"))[jcode]
J_IMG_SPLIT = pd.Series(J_SPLIT, index=cells.image_name.values).groupby(level=0).first()
tifs = g("*.tiff")
J_IMG = {os.path.splitext(os.path.basename(p))[0]: p for p in tifs
         if not os.path.basename(p).startswith("MB_") and "/masks/" not in p.replace("\\", "/") and "jackson" in p.lower()}
J_TRAIN = sorted(s for s in J_IMG_SPLIT.index[J_IMG_SPLIT == "train"] if s in J_IMG)
J_VAL = sorted(s for s in J_IMG_SPLIT.index[J_IMG_SPLIT == "val"] if s in J_IMG)
print(f"Jackson: {len(cells):,} cells, {len(jp)} patients; images train {len(J_TRAIN)}, val {len(J_VAL)}")
assert len(J_TRAIN) == 397 and len(J_VAL) == 39, "split differs from immuvis-jackson-train (397 / 39 images)"
del cells
gc.collect()

## ImmuVis code

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "comet_ml", "ruamel.yaml"], check=True)
import zipfile, urllib.request
from skimage.filters import butterworth

CODE = "/tmp/immuvis_code"
def fetch_code():
    try:
        urllib.request.urlretrieve("https://anonymous.4open.science/api/repo/ImmuVis-9173/zip", "/tmp/immuvis.zip")
        zipfile.ZipFile("/tmp/immuvis.zip").extractall(CODE)
        return "anonymous.4open.science/r/ImmuVis-9173"
    except Exception as e:
        print("anonymous release not available:", e)
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/dav3794/multiplex-image-model.git", CODE], check=True)
        return "github.com/dav3794/multiplex-image-model"
CODE_SOURCE = fetch_code()
root = CODE if os.path.isdir(f"{CODE}/multiplex_model") else os.path.dirname(glob.glob(f"{CODE}/**/multiplex_model", recursive=True)[0])
print("ImmuVis code from", CODE_SOURCE)

CLIP = 2.3
CROP, BATCH = 128, 16
TRAIN_MIN = float(os.environ.get("TRAIN_MIN", 180))
EVAL_EVERY = 500

def read_hwc(path, n_chan):
    im = tifffile.imread(path)
    if im.ndim == 3 and im.shape[0] == n_chan and im.shape[-1] != n_chan:
        im = np.moveaxis(im, 0, -1)
    assert im.shape[-1] == n_chan, (path, im.shape)
    return im.astype(np.float32)

def immuvis_pre(im_hwc, names, idx):
    x = np.arcsinh(np.stack([im_hwc[..., idx[c]] for c in names]) / COF)
    x = np.stack([butterworth(c, cutoff_frequency_ratio=0.2, high_pass=False) for c in x])
    return (np.clip(x, 0, CLIP) / CLIP).astype(np.float16)

## Preprocess, then train in a child process (as in `immuvis-jackson-train`)

In [ ]:
import psutil
jix = {m: i for i, m in enumerate(J_CHAN)}

def sysmem():
    vm = psutil.virtual_memory()
    return f"system RAM used {(vm.total - vm.available) / 2**30:.1f} of {vm.total / 2**30:.1f} GB"

def prep_jackson(s):
    return immuvis_pre(read_hwc(J_IMG[s], len(J_CHAN)), CH, jix)

def save_flat(names, path):
    arrs = Parallel(n_jobs=4)(delayed(prep_jackson)(s) for s in names)
    shapes = np.array([a.shape[1:] for a in arrs], np.int64)
    np.save(f"{path}_flat.npy", np.concatenate([a.reshape(len(CH), -1) for a in arrs], axis=1))
    np.save(f"{path}_shapes.npy", shapes)
    return len(arrs)

N_TR = save_flat(J_TRAIN, "/tmp/jtr")
N_VA = save_flat(J_VAL, "/tmp/jva")
from joblib.externals.loky import get_reusable_executor
get_reusable_executor().shutdown(wait=True)
gc.collect()
tick(f"Jackson preprocessed ({N_TR} train, {N_VA} val images, {len(CH)} channels); {sysmem()}")

SCRIPT = r"""
import os, sys, json, time, gc, traceback
cfg = json.loads(sys.argv[1])
import numpy as np, pandas as pd, torch, psutil
import torchvision.transforms.functional as TF
from torch.amp import autocast, GradScaler
sys.path.insert(0, cfg["root"])
from multiplex_model.modules import MultiplexAutoencoder
from multiplex_model.utils import (EncoderConfig, DecoderConfig, apply_channel_masking, apply_spatial_masking,
                                   ClampWithGrad, get_scheduler_with_warmup)
from multiplex_model.losses import beta_nll_loss
torch.backends.cudnn.enabled = False        # see immuvis-profile: cuDNN on a T4 ran ~6 s/step and leaked host RAM
OUT, CH, CLIP, CROP, BATCH = cfg["out"], cfg["channels"], cfg["clip"], cfg["crop"], cfg["batch"]
C = len(CH)
SH = list(range(cfg["n_shared"]))           # the 14 shared channels
EP = list(range(cfg["n_shared"], C))        # H3K27me3, pHH3
LOGF = open(f"{OUT}/train_progress.log", "a", buffering=1)
T0 = time.time()

def log(msg):
    vm = psutil.virtual_memory()
    LOGF.write(f"[{(time.time() - T0) / 60:6.1f} min] {msg} | RSS {psutil.Process().memory_info().rss / 2**30:.1f} GB, "
               f"system used {(vm.total - vm.available) / 2**30:.1f} GB, GPU0 {torch.cuda.memory_allocated(0) / 2**30:.1f} GB\n")

def load(path):
    flat = np.load(f"{path}_flat.npy", mmap_mode="r")
    shapes = np.load(f"{path}_shapes.npy")
    offs = np.r_[0, np.cumsum(shapes[:, 0] * shapes[:, 1])]
    return [flat[:, offs[i]:offs[i + 1]].reshape(C, *shapes[i]) for i in range(len(shapes))]

JTR, JVA = load("/tmp/jtr"), load("/tmp/jva")
dev = "cuda"
N_GPU = torch.cuda.device_count()
SEED = cfg["seed"]
IDS = torch.arange(C)
_rv = np.random.default_rng(7)
VCROPS = []
while len(VCROPS) < cfg["n_val_crops"]:
    x = JVA[_rv.integers(len(JVA))]
    H, W = x.shape[1:]
    if H < CROP or W < CROP:
        continue
    y0, x0 = _rv.integers(0, H - CROP + 1), _rv.integers(0, W - CROP + 1)
    VCROPS.append(np.asarray(x[:, y0:y0 + CROP, x0:x0 + CROP]))
VCROPS = torch.from_numpy(np.stack(VCROPS).astype(np.float32))
ENC = EncoderConfig(**cfg["enc_args"]).model_dump()
DEC = DecoderConfig(**cfg["dec_args"]).model_dump()
_rng = np.random.default_rng(SEED)
WIN = int(np.ceil(CROP * np.sqrt(2))) + 2

def train_batch():
    out = []
    for _ in range(BATCH):
        x = JTR[_rng.integers(len(JTR))]
        H, W = x.shape[1:]
        if H < WIN or W < WIN:
            x = np.pad(x, ((0, 0), (0, max(0, WIN - H)), (0, max(0, WIN - W))))
            H, W = x.shape[1:]
        y0, x0 = _rng.integers(0, H - WIN + 1), _rng.integers(0, W - WIN + 1)
        out.append(np.asarray(x[:, y0:y0 + WIN, x0:x0 + WIN]))
    t = torch.from_numpy(np.stack(out).astype(np.float32)).to(dev)
    res = []
    for b in range(BATCH):                       # RandomRotation(180, bilinear) -> crop -> RandomHorizontalFlip
        r = TF.center_crop(TF.rotate(t[b], float(_rng.uniform(-180, 180)), interpolation=TF.InterpolationMode.BILINEAR),
                           [CROP, CROP])
        res.append(torch.flip(r, dims=[-1]) if _rng.random() < 0.5 else r)
    return torch.stack(res)

def build():
    torch.manual_seed(SEED)
    net = MultiplexAutoencoder(num_channels=C, encoder_config=ENC, decoder_config=DEC).to(dev)
    return net, (torch.nn.DataParallel(net) if N_GPU > 1 else net)

@torch.no_grad()
def pixel_r(net, act, targets):
    # pixel r on the val crops of each target channel, predicted from the channels in act
    preds = []
    for b0 in range(0, len(VCROPS), 32):
        xb = VCROPS[b0:b0 + 32].to(dev)
        with autocast(device_type="cuda", dtype=torch.float16):
            lat = net.encode(xb[:, act], IDS[act].to(dev).expand(len(xb), -1))["output"]
            o = net.decode(lat, IDS[targets].to(dev).expand(len(xb), -1))
        preds.append(torch.sigmoid(o[..., 0].float()).cpu())
    p = torch.cat(preds).numpy()                 # [n, len(targets), H, W]
    rs = []
    for j, k in enumerate(targets):
        a, b = p[:, j].ravel(), VCROPS[:, k].numpy().ravel()
        rs.append(float(np.corrcoef(a, b)[0, 1]) if a.std() > 0 and b.std() > 0 else np.nan)
    return rs

def validate(net, with_loo):
    net.eval()
    epi = pixel_r(net, SH, EP)                   # the test-time task: both marks from the 14 shared channels
    loo = [pixel_r(net, [i for i in SH if i != k], [k])[0] for k in range(3, len(SH))] if with_loo else []
    net.train()
    return float(np.nanmean(epi)), epi, loo

def step_fn(model, img, opt, scaler, sched):
    ids = IDS.to(dev).expand(len(img), -1)
    img, ids, masked, act = apply_channel_masking(img, ids, 0.75, 0.5, apply_channel_subset_sampling=True)
    masked, _ = apply_spatial_masking(masked, 0.6, 8)
    with autocast(device_type="cuda", dtype=torch.float16):
        out = model(masked, act, ids)["output"]
    mi, logvar = out.float().unbind(dim=-1)
    loss = beta_nll_loss(img, torch.sigmoid(mi), ClampWithGrad.apply(logvar, -15.0, 15.0), beta=0.5)
    if not torch.isfinite(loss):
        opt.zero_grad(set_to_none=True)
        return float("nan")
    scaler.scale(loss).backward()
    scaler.unscale_(opt)
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    scaler.step(opt)
    scaler.update()
    opt.zero_grad(set_to_none=True)
    sched.step()
    return float(loss.item())

try:
    log(f"child started: {len(JTR)} train / {len(JVA)} val images, {len(VCROPS)} val crops, {C} channels, {N_GPU} GPUs")
    net, model = build()
    opt = torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=1e-4)
    scaler = GradScaler()
    sched = get_scheduler_with_warmup(opt, 100, 10_000, final_lr=1e-5, peak_lr=5e-4, type="cosine")
    for _ in range(5):
        step_fn(model, train_batch(), opt, scaler, sched)
    torch.cuda.synchronize()
    t_start = time.time()
    for _ in range(30):
        step_fn(model, train_batch(), opt, scaler, sched)
    torch.cuda.synchronize()
    sec = (time.time() - t_start) / 30
    t_val = time.time()
    validate(net, True)
    sec_val = time.time() - t_val
    EVAL_EVERY = cfg["eval_every"]
    TOTAL = int(cfg["train_min"] * 60 / (sec + sec_val / EVAL_EVERY))
    log(f"{sec:.3f} s/step, validation {sec_val:.0f} s -> {TOTAL} steps in {cfg['train_min']:.0f} min")
    net, model = build()
    opt = torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=1e-4)
    scaler = GradScaler()
    sched = get_scheduler_with_warmup(opt, int(0.1 * TOTAL), TOTAL - int(0.1 * TOTAL), final_lr=1e-5, peak_lr=5e-4, type="cosine")
    LOGT, best, losses, n_nan, n_eval = [], -np.inf, [], 0, 0
    t_train = time.time()
    model.train()
    for step in range(1, TOTAL + 1):
        l = step_fn(model, train_batch(), opt, scaler, sched)
        n_nan += int(not np.isfinite(l))
        losses.append(l)
        if step % EVAL_EVERY == 0 or step == TOTAL:
            with_loo = n_eval % 4 == 0 or step == TOTAL
            n_eval += 1
            vr, epi, loo = validate(net, with_loo)
            row = dict(step=step, minutes=(time.time() - t_train) / 60, train_loss=float(np.nanmean(losses)),
                       val_epi_pixel_r=vr, nan_steps=n_nan, lr=sched.get_last_lr()[0],
                       **{f"val_r__{CH[k]}": epi[j] for j, k in enumerate(EP)})
            if loo:
                row.update(val_loo_pixel_r_shared=float(np.nanmean(loo)),
                           **{f"val_loo_r__{CH[k]}": loo[k - 3] for k in range(3, len(SH))})
            LOGT.append(row)
            losses = []
            if vr > best:
                best = vr
                torch.save({"model_state_dict": {k: v.detach().cpu().clone() for k, v in net.state_dict().items()},
                            "model_config": net.get_architecture_config(), "channels": CH, "n_shared": len(SH),
                            "clip": CLIP, "code_source": cfg["code_source"], "best_val_epi_pixel_r": best,
                            "val_r_at_best": {CH[k]: epi[j] for j, k in enumerate(EP)}, "step": step,
                            "total_steps": TOTAL, "train_minutes": cfg["train_min"], "train_cohort": "jackson",
                            "n_train_images": len(JTR), "n_val_images": len(JVA)},
                           f"{OUT}/epi_immuvis_jackson_best.pth")
            pd.DataFrame(LOGT).to_csv(f"{OUT}/immuvis_training_log.csv", index=False)
            log(f"step {step}/{TOTAL}: train loss {row['train_loss']:.4f}, val r H3K27me3 {epi[0]:.3f}, pHH3 {epi[1]:.3f} "
                f"(best mean {best:.3f})" + (f", shared LOO r {np.nanmean(loo):.3f}" if loo else ""))
    json.dump({"total_steps": TOTAL, "s_per_step": sec, "best_val_epi_pixel_r": best, "nan_steps": n_nan},
              open(f"{OUT}/child_done.json", "w"))
    log("child done")
except Exception:
    LOGF.write("CHILD EXCEPTION\n" + traceback.format_exc())
    raise
"""
open("/tmp/train_immuvis.py", "w").write(SCRIPT)
CFG = dict(root=root, out=OUT, channels=CH, n_shared=len(NUC) + len(MARKERS), clip=CLIP, crop=CROP, batch=BATCH,
           seed=SEED, train_min=TRAIN_MIN, eval_every=EVAL_EVERY, n_val_crops=128, code_source=CODE_SOURCE,
           enc_args=dict(ma_layers_blocks=[4], ma_embedding_dims=[16], pm_layers_blocks=[4, 4, 4],
                         pm_embedding_dims=[192, 384, 768], hyperkernel=dict(kernel_size=1, padding=0, stride=1, use_bias=True)),
           dec_args=dict(decoded_embed_dim=512, num_blocks=1,
                         hyperkernel=dict(kernel_size=1, padding=0, stride=1, use_bias=True)))
PROG = f"{OUT}/train_progress.log"
open(PROG, "w").close()
child = subprocess.Popen([sys.executable, "/tmp/train_immuvis.py", json.dumps(CFG)],
                         stdout=open(f"{OUT}/child_stdout.log", "w"), stderr=subprocess.STDOUT)
seen, t_launch = 0, time.time()
while child.poll() is None:
    time.sleep(60)
    lines = open(PROG).read().splitlines()
    for ln in lines[seen:]:
        print(ln, flush=True)
    seen = len(lines)
    try:
        crss = psutil.Process(child.pid).memory_info().rss / 2**30
    except psutil.Error:
        crss = float("nan")
    if int(time.time() - t_launch) % 600 < 60:
        print(f"[monitor {(time.time() - t_launch) / 60:.0f} min] child RSS {crss:.1f} GB; {sysmem()}", flush=True)
lines = open(PROG).read().splitlines()
for ln in lines[seen:]:
    print(ln, flush=True)
rc = child.returncode
print(f"child exit code {rc}" + (f" (killed by signal {signal.Signals(-rc).name})" if rc < 0 else ""), flush=True)
print("child stdout tail:\n" + "".join(open(f"{OUT}/child_stdout.log").readlines()[-30:]), flush=True)
json.dump({"code_source": CODE_SOURCE, "train_minutes": TRAIN_MIN, "child_exit_code": rc, "channels": CH,
           "minutes": (time.time() - T0) / 60, "n_jackson_train_images": N_TR, "n_jackson_val_images": N_VA,
           **(json.load(open(f"{OUT}/child_done.json")) if os.path.exists(f"{OUT}/child_done.json") else {})},
          open(f"{OUT}/run_info.json", "w"))
assert rc == 0 and os.path.exists(f"{OUT}/epi_immuvis_jackson_best.pth"), "training child failed"
tick("ImmuVis trained")